# Data Imports

In [319]:
import pandas as pd
import numpy as np
import json
import math

In [320]:
# import Artists dataset from Getty, csv file downloaded 10/22/26
Artists = pd.read_csv("Artists.csv")
Artists.head(5)

,x,name,bio,nationality,type
0,http://vocab.getty.edu/ulan/500771934,"Aa, Andries van der","North Netherlandish painter, 1743-1776",North Netherlandish,painters (artists)
1,http://vocab.getty.edu/ulan/500557119,"Aabdaoui, Latif","Moroccan graphic artist, 1958-",Moroccan,graphic artists
2,http://vocab.getty.edu/ulan/500632715,"Aa, Ben","Dutch painter, born 1951",Dutch (culture or style),painters (artists)
3,http://vocab.getty.edu/ulan/500546545,"Aaberg, Gunhild","Danish ceramicist, 1939-",Danish (culture or style),ceramicists
4,http://vocab.getty.edu/ulan/500108834,"Aabergh, Gösta","Norwegian architect, active mid-late 20th century",Norwegian (culture),architects


In [321]:
# Import Art Institute of Chicago dataset, jsonl file from 10/22/26
Chicago = pd.read_json("AIofChicago.jsonl", lines = True) 
Chicago.head(5)

,id,title,main_reference_number,department_title,artist_title
0,4,Priest and Boy,1880.1,Prints and Drawings,Lawrence Carmichael Earle
1,9,"Interior of St. Mark's, Venice",1887.232,Arts of the Americas,David Dalhoff Neal
2,11,Self-Portrait,1887.234,Arts of the Americas,Walter Shirlaw
3,16,The Fall of the Giants,1887.249,Prints and Drawings,Salvator Rosa
4,19,"View of Ponte Lugano on the Anio, from Views o...",1887.252,Prints and Drawings,Giovanni Battista Piranesi


In [ ]:
# Import The Met's dataset from their github
Met = pd.read_csv("https://github.com/metmuseum/openaccess/raw/refs/heads/master/MetObjects.csv") 

/var/folders/sj/4z_czbbd63l7mskgh4qnw0y80000gn/T/ipykernel_39161/1058838771.py:2: DtypeWarning: Columns (5,7,10,11,12,13,14,15,34,35,36,37,38,39,40,41,42,43,44,45,46) have mixed types. Specify dtype option on import or set low_memory=False.
  Met = pd.read_csv("https://github.com/metmuseum/openaccess/raw/refs/heads/master/MetObjects.csv")


The columns warned as having mixed types are as follows:

In [323]:
pd.DataFrame({"type": Met.dtypes, "colnum": np.arange(0, len(Met.columns))}).reset_index().loc[[5,7,10,11,12,13,14,15,34,35,36,37,38,39,40,41,42,43,44,45,46]]

,index,type,colnum
5,Gallery Number,object,5
7,AccessionYear,object,7
10,Culture,object,10
11,Period,object,11
12,Dynasty,object,12
13,Reign,object,13
14,Portfolio,object,14
15,Constituent ID,object,15
34,Geography Type,object,34
35,City,object,35


We will focus on a subset of the Met dataset, with variables "Object Number", "Department", "Accession Year", "Object Name", "Title", and "Artist Display Name" chosen to match the Art Institute of Chicago dataset. Because of this, "Accession Year" is the only variable we have to check data type in. 

In [324]:
Met = Met[["Object Number", "Department", "AccessionYear", "Object Name", "Title", "Artist Display Name"]]
Met.head(5)

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name
0,1979.486.1,The American Wing,1979.0,Coin,One-dollar Liberty Head Coin,James Barton Longacre
1,1980.264.5,The American Wing,1980.0,Coin,Ten-dollar Liberty Head Coin,Christian Gobrecht
2,67.265.9,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN
3,67.265.10,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN
4,67.265.11,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN


In [325]:
# inspect AccessionYear
Met = Met.astype({"AccessionYear": "str"})
Met["AccessionYear"].unique()

array(['1979.0', '1980.0', '1967.0', '1916.0', '1909.0', '1964.0',
       '1970.0', '1904.0', '1976.0', '1938.0', '1918.0', '1946.0',
       '1960.0', '1910.0', '1922.0', '1923.0', '1930.0', '1934.0',
       '1936.0', '1937.0', '1939.0', '1941.0', '1947.0', '1948.0',
       '1949.0', '1971.0', '1984.0', '1989.0', '1985.0', '1952.0',
       '1968.0', '1919.0', 'nan', '1978.0', '1972.0', '1924.0', '1983.0',
       '1950.0', '1925.0', '1926.0', '1931.0', '1932.0', '1951.0',
       '1953.0', '2006.0', '1969.0', '1975.0', '2025.0', '1981.0',
       '1986.0', '1959.0', '1944.0', '1982.0', '1945.0', '1888.0',
       '1911.0', '1957.0', '1943.0', '1935.0', '1917.0', '1973.0',
       '1940.0', '1913.0', '1962.0', '1914.0', '1966.0', '1954.0',
       '1920.0', '1933.0', '2014.0', '1928.0', '1901.0', '1958.0',
       '1965.0', '1883.0', '1902.0', '1987.0', '1974.0', '1977.0',
       '1915.0', '1942.0', '1912.0', '1927.0', '2023.0', '1896.0',
       '1892.0', '1877.0', '1906.0', '1955.0', '1961.0'

In [326]:
# the beginning of Object Number can be used to find corrected AccessionYear
Met[Met["AccessionYear"] == "2028"]

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name
489224,2024.626.7,Photographs,2028,Photograph,Zuma #21,John Divola


In [327]:
# Not all Object Numbers have the full Accession Year as the first split
Met["Object Number"].str.split(".", n = 1, expand = True)[0].head()

0    1979
1    1980
2      67
3      67
4      67
Name: 0, dtype: object

Within Accession Year we see some double values (ex: 1979.0), some integer values (ex: 1941), and some date values (ex: 1973-05-08). We also see some values that are not possible, since they are in the future, such as 2028. When we check the corresponding Object Number, we can see that this piece was likely aquired in 2024. Using just the Accession Number is also not a solution, since some values leave out the century and just have the year (ex: 1967 as 67).

In [328]:
# create new column with cleaned Acession Year as a numeric value
Met["Cleaned_AY"] = Met["AccessionYear"].str.split(".", n = 1, expand = True)[0].str.split("-", n = 1, expand = True)[0]
Met = Met.astype({"Cleaned_AY": "float64"})
Met.head()

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY
0,1979.486.1,The American Wing,1979.0,Coin,One-dollar Liberty Head Coin,James Barton Longacre,1979.0
1,1980.264.5,The American Wing,1980.0,Coin,Ten-dollar Liberty Head Coin,Christian Gobrecht,1980.0
2,67.265.9,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0
3,67.265.10,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0
4,67.265.11,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0


In [329]:
# we find 223 instances of an Accession Year in the future
Met[Met["Cleaned_AY"] > 2026]

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY
489223,2024.626.6,Photographs,2027,Photograph,Zuma #20,John Divola,2027.0
489224,2024.626.7,Photographs,2028,Photograph,Zuma #21,John Divola,2028.0
489225,2024.626.8a–c,Photographs,2029,Photograph,"51st Street, 18th June 2010, 1.28.45 pm",Paul Graham,2029.0
489226,2024.626.9,Photographs,2030,Photograph,"The Polo Grounds, New York",Irving Penn,2030.0
489227,2024.626.10,Photographs,2031,Photograph,"30,000 kVA Steam Turbine [Power Station] Ruhrc...",Albert Renger-Patzsch,2031.0
...,...,...,...,...,...,...,...
490728,2024.626.224,Photographs,2056,Photograph,New York,Garry Winogrand,2056.0
490729,2024.626.225,Photographs,2056,Photograph,"Salinas, Kansas",Garry Winogrand,2056.0
490730,2024.626.226,Photographs,2056,Photograph,"Candlestick Park, San Francisco",Garry Winogrand,2056.0
490731,2024.626.227,Photographs,2056,Photograph,London,Garry Winogrand,2056.0


___

In [ ]:
# KEEP
Met["Cleaned_AY"].case_when(
    caselist = [
        (Met["Cleaned_AY"] > 2026, Met["AYfromON"])#, # when year is greater than 2026, replace with year from Object Number
        #(Met["Cleaned_AY"])
    ]
)

array([1979.0, 1980.0, 1967.0, 1916.0, 1909.0, 1964.0, 1970.0, 1904.0,
       1976.0, 1938.0, 1918.0, 1946.0, 1960.0, 1910.0, 1922.0, 1923.0,
       1930.0, 1934.0, 1936.0, 1937.0, 1939.0, 1941.0, 1947.0, 1948.0,
       1949.0, 1971.0, 1984.0, 1989.0, 1985.0, 1952.0, 1968.0, 1919.0,
       nan, 1978.0, 1972.0, 1924.0, 1983.0, 1950.0, 1925.0, 1926.0,
       1931.0, 1932.0, 1951.0, 1953.0, 2006.0, 1969.0, 1975.0, 2025.0,
       1981.0, 1986.0, 1959.0, 1944.0, 1982.0, 1945.0, 1888.0, 1911.0,
       1957.0, 1943.0, 1935.0, 1917.0, 1973.0, 1940.0, 1913.0, 1962.0,
       1914.0, 1966.0, 1954.0, 1920.0, 1933.0, 2014.0, 1928.0, 1901.0,
       1958.0, 1965.0, 1883.0, 1902.0, 1987.0, 1974.0, 1977.0, 1915.0,
       1942.0, 1912.0, 1927.0, 2023.0, 1896.0, 1892.0, 1877.0, 1906.0,
       1955.0, 1961.0, 1994.0, 1907.0, 1929.0, 2018.0, 1921.0, 1908.0,
       1963.0, 2010.0, 2000.0, 1988.0, 2021.0, 1956.0, 2012.0, 2013.0,
       2002.0, 1999.0, 1894.0, 1897.0, 1993.0, 1893.0, 1992.0, 1996.0,
       18

In [ ]:
# Add Accession Year from Object Number column - INCOMPLETE
Met["AYfromON"] = Met["Object Number"].str.split(".", n = 1, expand = True)[0]
Met.head()

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY,AYfromON
0,1979.486.1,The American Wing,1979.0,Coin,One-dollar Liberty Head Coin,James Barton Longacre,1979.0,1979
1,1980.264.5,The American Wing,1980.0,Coin,Ten-dollar Liberty Head Coin,Christian Gobrecht,1980.0,1980
2,67.265.9,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67
3,67.265.10,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67
4,67.265.11,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67


In [318]:
Met[Met["Object Number"].str.split(".", n = 1, expand = True)[0].str.isnumeric() == False]["AYfromON"].unique()

array(['68', '69', 'X', ..., 'T53 1914', 'T5 A4 1915',
       'M4 M49 1911 copy 2'], shape=(1616,), dtype=object)

In [315]:
Met["AYfromON"] = Met["AYfromON"].case_when(
    caselist = [
        (Met["Object Number"].str.split(".", expand = True)[0].str.isnumeric() == False, Met["Object Number"].str.split(".", expand = True)[1]),
        (Met["Object Number"].str.split(".", expand = True)[1].str.isnumeric() == False, Met["Object Number"].str.split(".", expand = True)[2]),
        (Met["Object Number"].str.split(".", expand = True)[1].str.isnumeric() == False, Met["Object Number"].str.split(".", expand = True)[2]),
    ]
)

In [352]:
Met["Object Number"].str.findall("(\\d+)")

0           [1979, 486, 1]
1           [1980, 264, 5]
2             [67, 265, 9]
3            [67, 265, 10]
4            [67, 265, 11]
                ...       
502944    [69, 503, 9, 22]
502945    [69, 503, 9, 23]
502946    [69, 503, 9, 24]
502947    [59, 654, 22, 1]
502948    [59, 654, 22, 2]
Name: Object Number, Length: 502949, dtype: object

In [299]:
Met["Object Number"].str.split(".", expand = True)

#.str.isnumeric() == False

,0,1,2,3,4,5,6,7,8,9,10,11
0,1979,486,1,None,None,None,None,None,None,None,None,None
1,1980,264,5,None,None,None,None,None,None,None,None,None
2,67,265,9,None,None,None,None,None,None,None,None,None
3,67,265,10,None,None,None,None,None,None,None,None,None
4,67,265,11,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
502944,69,503,9(22),None,None,None,None,None,None,None,None,None
502945,69,503,9(23),None,None,None,None,None,None,None,None,None
502946,69,503,9(24),None,None,None,None,None,None,None,None,None
502947,59,654,22(1),None,None,None,None,None,None,None,None,None


In [316]:
Met[np.isnan(Met["Cleaned_AY"])]

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY,AYfromON
123,Inst.68.4.1,The American Wing,nan,Architectural elements,"Architectural elements from Halsted House, Rye...",NaN,NaN,68
130,Inst.69.8.1,The American Wing,nan,Lamp shade,Lamp Shade,NaN,NaN,69
131,Inst.69.8.2,The American Wing,nan,Lamp shade,Lamp Shade,NaN,NaN,69
237,Inst.X.6.1,The American Wing,nan,Furniture hardware,Furniture Hardware,NaN,NaN,X
390,Inst.1983.2,The American Wing,nan,Bell pull,Bell Pull,NaN,NaN,1983
...,...,...,...,...,...,...,...,...
502901,Museum Accession.32,Drawings and Prints,nan,Print,"The Jewish Bride, from The American Art Review...",Rembrandt (Rembrandt van Rijn)|Estes and Lauri...,NaN,32
502906,Museum Accession.33,Drawings and Prints,nan,Print,"Joan of Arc's Entry into New Orleans, Suppleme...",Aubrey Vincent Beardsley|Carl Hentschel,NaN,33
502920,NA6840.M4 M49 1911 copy 2,The Libraries,nan,NaN,Mosaic curtain for the National Theatre of Mexico,Tiffany Studios,NaN,M4 M49 1911 copy 2
502921,68.754.1(1-10),Drawings and Prints,nan,Portfolio,Torn Drawings,Leon Polk Smith|Galerie Chalette,NaN,68
